In [1]:
!pip install faiss-cpu sentence-transformers transformers torch pypdf

In [2]:
import os
import json
import faiss
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# ── 1. Reuse your existing chunk_text function ──────────────────────────────
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

def chunk_text(text, chunk_size=400, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
        if len(words) - start < 50:
            break
    return chunks

# ── 2. Load all PDFs and chunk them ────────────────────────────────────────
def load_documents(input_dir="./documents"):
    all_chunks = []   # stores raw text of each chunk
    chunk_metadata = []  # stores which file each chunk came from

    if not os.path.exists(input_dir):
        print(f"Error: '{input_dir}' not found.")
        return all_chunks, chunk_metadata

    for filename in os.listdir(input_dir):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(input_dir, filename)
            print(f"Loading: {filename}")
            text = extract_text_from_pdf(pdf_path)
            chunks = chunk_text(text, chunk_size=400, overlap=50)
            for idx, chunk in enumerate(chunks):
                all_chunks.append(chunk)
                chunk_metadata.append({
                    "filename": filename,
                    "chunk_index": idx,
                    "total_chunks": len(chunks)
                })
            print(f"  → {len(chunks)} chunks created")

    print(f"\nTotal chunks across all PDFs: {len(all_chunks)}")
    return all_chunks, chunk_metadata

# ── 3. Build FAISS index ────────────────────────────────────────────────────
def build_faiss_index(chunks, index_path="faiss_index", metadata_path="chunk_metadata.json"):
    """
    Converts text chunks into embeddings and stores them in a FAISS index.
    - SentenceTransformer converts text → 384-dim float vectors
    - FAISS stores these vectors for fast similarity search
    """
    print("Loading embedding model...")
    # This small model (90MB) converts text into numerical vectors
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    print("Generating embeddings for all chunks...")
    # Shape: (num_chunks, 384) — each chunk becomes a 384-dimensional vector
    embeddings = embedder.encode(chunks, show_progress_bar=True, convert_to_numpy=True)

    # Normalize vectors so cosine similarity = dot product (faster search)
    faiss.normalize_L2(embeddings)

    # Create FAISS index — IndexFlatIP = exact search using inner product
    dimension = embeddings.shape[1]  # 384
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    # Save index to disk so you don't need to rebuild every time
    faiss.write_index(index, index_path)

    # Save metadata (which chunk belongs to which file)
    with open(metadata_path, "w") as f:
        json.dump({"chunks": chunks, "metadata": chunk_metadata}, f)

    print(f"\n✅ FAISS index built with {index.ntotal} vectors")
    print(f"   Saved index → '{index_path}'")
    print(f"   Saved metadata → '{metadata_path}'")
    return index, embedder

# ── Run ─────────────────────────────────────────────────────────────────────
all_chunks, chunk_metadata = load_documents("./documents")
faiss_index, embedder = build_faiss_index(all_chunks)

Loading: 3545008.3545087.pdf
  → 25 chunks created
Loading: applsci-12-02160-v2.pdf
  → 19 chunks created
Loading: 3458817.3476223.pdf
  → 42 chunks created
Loading: Paper1.pdf
  → 12 chunks created
Loading: DBA_Residency_hsampatirao.pdf
  → 2 chunks created
Loading: Paper2.pdf
  → 11 chunks created
Loading: Hariprasad_Sampatirao_Draft_Chapter1.pdf
  → 25 chunks created
Loading: atc23-weng.pdf
  → 33 chunks created
Loading: 3638757.pdf
  → 54 chunks created
Loading: Walsh_Dissertation_Data Readiness.pdf
  → 15 chunks created

Total chunks across all PDFs: 238
Loading embedding model...
Generating embeddings for all chunks...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


✅ FAISS index built with 238 vectors
   Saved index → 'faiss_index'
   Saved metadata → 'chunk_metadata.json'


In [3]:
def retrieve_relevant_chunks(query, index, embedder, chunks, metadata, top_k=3):
    """
    Given a query, finds the top_k most relevant chunks from the FAISS index.

    How it works:
    1. Embed the query into the same 384-dim vector space as the chunks
    2. FAISS searches for the nearest vectors (most similar chunks)
    3. Return those chunks as context
    """
    # Embed the query
    query_vector = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vector)

    # Search FAISS — returns distances and indices of top_k matches
    distances, indices = index.search(query_vector, top_k)

    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        results.append({
            "rank": rank + 1,
            "score": float(dist),        # cosine similarity score (0-1, higher = better)
            "chunk": chunks[idx],
            "source": metadata[idx]["filename"],
            "chunk_index": metadata[idx]["chunk_index"]
        })
        print(f"  [{rank+1}] Score: {dist:.3f} | Source: {metadata[idx]['filename']} "
              f"(chunk {metadata[idx]['chunk_index']})")

    return results

# Quick test
print("Testing retrieval...")
test_results = retrieve_relevant_chunks(
    query="What is data readiness?",
    index=faiss_index,
    embedder=embedder,
    chunks=all_chunks,
    metadata=chunk_metadata,
    top_k=3
)

Testing retrieval...
  [1] Score: 0.414 | Source: Walsh_Dissertation_Data Readiness.pdf (chunk 0)
  [2] Score: 0.390 | Source: 3638757.pdf (chunk 49)
  [3] Score: 0.372 | Source: Walsh_Dissertation_Data Readiness.pdf (chunk 3)


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_tinyllama():
    """
    Loads TinyLlama directly from HuggingFace.
    We use the same model family as your finetuning setup for consistency.
    """
    MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    print(f"Loading TinyLlama from HuggingFace...")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,    # float16 to save GPU memory (same as finetuning)
        device_map="auto"             # automatically uses GPU if available
    )
    model.eval()  # set to inference mode — disables dropout etc.
    print("✅ TinyLlama loaded!")
    return tokenizer, model

tokenizer, llm_model = load_tinyllama()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading TinyLlama from HuggingFace...
✅ TinyLlama loaded!


In [5]:
def generate_answer(prompt, tokenizer, model, max_new_tokens=300):
    """
    Runs TinyLlama on a prompt and returns generated text.
    Uses TinyLlama's chat template for best results.
    """
    # TinyLlama uses a specific chat format — wrapping in it improves quality
    chat_prompt = f"<|system|>You are a helpful assistant that answers questions based on provided context.</s>\n<|user|>{prompt}</s>\n<|assistant|>"

    inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():  # no gradient tracking needed for inference
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,       # controls randomness (0=deterministic, 1=creative)
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens (skip the input prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def rag_qa(question, index, embedder, chunks, metadata, tokenizer, model, top_k=3):
    """
    Full RAG pipeline for Q&A:
    Retrieve → Augment prompt → Generate answer
    """
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    # Step 1: RETRIEVE relevant chunks
    print("\nRetrieving relevant chunks...")
    results = retrieve_relevant_chunks(question, index, embedder, chunks, metadata, top_k)

    # Step 2: AUGMENT — build context from retrieved chunks
    context = "\n\n---\n\n".join([
        f"[Source: {r['source']}, chunk {r['chunk_index']}]\n{r['chunk']}"
        for r in results
    ])

    # Step 3: GENERATE — feed context + question to TinyLlama
    prompt = f"""Based on the following context from research documents, answer the question.

Context:
{context}

Question: {question}

Answer clearly and concisely based only on the context provided."""

    print("\nGenerating answer...")
    answer = generate_answer(prompt, tokenizer, model)

    print(f"\nAnswer:\n{answer}")
    return answer


def rag_summarize(filename, index, embedder, chunks, metadata, tokenizer, model):
    """
    Full RAG pipeline for Summarization:
    Retrieve all chunks from a specific file → Generate summary
    """
    print(f"\n{'='*60}")
    print(f"Summarizing: {filename}")
    print(f"{'='*60}")

    # Get ALL chunks from this specific file
    file_chunks = [
        chunks[i] for i, m in enumerate(metadata)
        if m["filename"] == filename
    ]

    if not file_chunks:
        print(f"No chunks found for '{filename}'")
        return

    print(f"Found {len(file_chunks)} chunks for this document")

    # For summarization, use first 5 chunks to stay within token limits
    # (covers intro, methodology, key findings usually)
    context = "\n\n---\n\n".join(file_chunks[:5])

    prompt = f"""Based on the following excerpts from '{filename}', provide a structured summary.

Document excerpts:
{context}

Provide a summary covering:
1. Main topic and purpose
2. Key findings or arguments
3. Conclusions"""

    print("\nGenerating summary...")
    summary = generate_answer(prompt, tokenizer, model, max_new_tokens=400)

    print(f"\nSummary:\n{summary}")
    return summary

In [6]:
# ── Load saved index (if restarting kernel) ─────────────────────────────────
# Uncomment these lines if you need to reload after kernel restart:
# faiss_index = faiss.read_index("faiss_index")
# with open("chunk_metadata.json") as f:
#     saved = json.load(f)
#     all_chunks = saved["chunks"]
#     chunk_metadata = saved["metadata"]
# embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ── Q&A ─────────────────────────────────────────────────────────────────────
answer = rag_qa(
    question="What is data readiness and why does it matter?",
    index=faiss_index,
    embedder=embedder,
    chunks=all_chunks,
    metadata=chunk_metadata,
    tokenizer=tokenizer,
    model=llm_model,
    top_k=3
)

# ── Summarization ────────────────────────────────────────────────────────────
summary = rag_summarize(
    filename="Walsh_Dissertation_Data Readiness.pdf",
    index=faiss_index,
    embedder=embedder,
    chunks=all_chunks,
    metadata=chunk_metadata,
    tokenizer=tokenizer,
    model=llm_model
)

Token indices sequence length is longer than the specified maximum sequence length for this model (3161 > 2048). Running this sequence through the model will result in indexing errors



Question: What is data readiness and why does it matter?

Retrieving relevant chunks...
  [1] Score: 0.374 | Source: 3638757.pdf (chunk 49)
  [2] Score: 0.349 | Source: Walsh_Dissertation_Data Readiness.pdf (chunk 0)
  [3] Score: 0.345 | Source: Walsh_Dissertation_Data Readiness.pdf (chunk 3)

Generating answer...


This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.



Answer:
1412sing16singquotprovPro,quot,sing,1,1,Sh,5562prov,Apppro2Tsing1Dcontext514L7452712Singand c1ScontextCDCC2andPro,V,GD3LogG1C,D,3C1R21,V6LSingprovPro1NeShHV2VCviCCCommDV,S,Ch3DCSingPCCDVCVSCSVScontextCNeProMeSSingShLin2UniversH,C2CLCVCGBVLProVVDD141422C2VC141ProWSC ShD2CPSh2639ShDSShSh18DNLPShS9ShRVRGVDshLinDGNeVV2Pres,69SingShFShVShVSShShCommshDHCSingIndCVShShChRSingShSingV2VVSing2ShSh2LSh2SuperN2S4SemRedComShGuVPolSingColSingCCSingSingFSing4HyRe5GSingVDSingProVCCEx5SingMusVSingDSSingSingF3V7SingGSingDC

Summarizing: Walsh_Dissertation_Data Readiness.pdf
Found 15 chunks for this document

Generating summary...

Summary:
9 for Mer A09 E126 A A A Pred Emp 16 A S T Est For2 Advanced L A T212 Anal E129 Pro, Parts, Wal A1 Dis Est W Sub, Pre S, T2247.2302 Mod Com2 Super Wal N1 Ant for Wal Anal00 Dis704231418 M2012 Supp3 W2 Wal F49,23 Preds Econom For E W12 for81038 S110 Wal290721 Wal­2 A222023 For for pur A63012for Preds 12 for29 Sal A2142022311 Wal Wal1178 Ad A Wal Supp Pres Wal E